Assignmnent:
1. Load 5mn bars for US stock
2. Plot terminal distributions from time t to EOD
3. Compare with risk-neutral probabilities
4. Backtest 0DTE Condor Strategy

In [44]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import sys
import plotly.express as px
import plotly.graph_objects as go
from time import time
from tqdm import tqdm
from itertools import product
import pytz
from datetime import datetime as dt

sys.path.append('../../')

In [52]:
PERIOD = 5    # minutes
DAILY_PERIODS = 6.5 * 60 / PERIOD    # number of trading periods in a day
ANNUAL_FACTOR = 252 * DAILY_PERIODS
HOURS_PER_YEAR = 252 * 10    # used to calculate option price (arbitrary, this determines the volatility)
STOCK = 'SPY'
TZ_EXCHANGE = 'America/Chicago'
LAST_BAR_TIME = '14:55:00'    # our data have bars until 3PM - PERIOD minutes before EOD

### Load Intraday Bar Data

In [3]:
# load OHLCV data from market_data/intraday/
ohlcv = pd.read_csv(f'../../market_data/intraday/{STOCK}_{PERIOD}mn.csv')
ohlcv = ohlcv.set_index('Date')
ohlcv.index = pd.to_datetime(ohlcv.index)
# convert UTC data to local time: Chicago time
ohlcv.index = ohlcv.index.tz_convert(TZ_EXCHANGE)
ohlcv.tail()

,Open,High,Low,Close,Volume
Date,,,,,
2026-03-17 14:35:00-05:00,671.00,671.14,670.94,670.99,11439.0
2026-03-17 14:40:00-05:00,670.99,671.15,670.75,670.86,18665.0
2026-03-17 14:45:00-05:00,670.87,671.21,670.85,671.18,23100.0
2026-03-17 14:50:00-05:00,671.18,671.28,670.58,671.21,44420.0
2026-03-17 14:55:00-05:00,671.20,671.20,670.47,670.73,73989.0


In [5]:
# keep only Close prices
# add each row's day expiry timestamp and its corresponding close

# close-only frame
df_price = ohlcv[['Close']].copy()

# last Date index for each trading day
expiry_by_day = df_price.index.to_series().groupby(df_price.index.normalize()).max()

# add expiry timestamp and close at expiry
df_price['Expiry'] = df_price.index.normalize().map(expiry_by_day)
df_price['Close_At_Expiry'] = df_price['Expiry'].map(df_price['Close'])

In [49]:
# add a column for the time to expiry
df_price['hours_to_exp'] = (df_price['Expiry'] - df_price.index).dt.total_seconds() / 3600
df_price.tail()

,Close,Expiry,Close_At_Expiry,time_to_expiry,hours_to_exp
Date,,,,,
2026-03-17 14:35:00-05:00,670.99,2026-03-17 14:55:00-05:00,670.73,0.333333,0.333333
2026-03-17 14:40:00-05:00,670.86,2026-03-17 14:55:00-05:00,670.73,0.250000,0.250000
2026-03-17 14:45:00-05:00,671.18,2026-03-17 14:55:00-05:00,670.73,0.166667,0.166667
2026-03-17 14:50:00-05:00,671.21,2026-03-17 14:55:00-05:00,670.73,0.083333,0.083333
2026-03-17 14:55:00-05:00,670.73,2026-03-17 14:55:00-05:00,670.73,0.000000,0.000000


In [54]:
# select a time in the morning
trade_time = '10:00:00'
hours_to_expiry = (dt.strptime(LAST_BAR_TIME, '%H:%M:%S') - dt.strptime(trade_time, '%H:%M:%S')).total_seconds() / 3600

# keep only rows at trade_time
df_trade = df_price[df_price.index.strftime('%H:%M:%S') == trade_time]
df_trade = df_trade[df_trade['hours_to_exp'] >= 0.99*hours_to_expiry]    # remove days where market closes early
df_trade['time_to_expiry'] = df_trade['hours_to_exp']/HOURS_PER_YEAR
df_trade.tail()

,Close,Expiry,Close_At_Expiry,time_to_expiry,hours_to_exp
Date,,,,,
2026-03-11 10:00:00-05:00,677.85,2026-03-11 14:55:00-05:00,676.35,0.001951,4.916667
2026-03-12 10:00:00-05:00,668.15,2026-03-12 14:55:00-05:00,666.05,0.001951,4.916667
2026-03-13 10:00:00-05:00,665.10,2026-03-13 14:55:00-05:00,662.31,0.001951,4.916667
2026-03-16 10:00:00-05:00,669.06,2026-03-16 14:55:00-05:00,668.96,0.001951,4.916667
2026-03-17 10:00:00-05:00,672.50,2026-03-17 14:55:00-05:00,670.73,0.001951,4.916667


In [ ]:
# plot the terminal distribution with plotly
df_trade['Return(%)'] = 100*(np.log(df_trade['Close_At_Expiry']/df_trade['Close']))
fig = px.histogram(df_trade, x='Return(%)', nbins=100)
fig.show()

### Black-Scholes model

In [51]:
# volatility as a parabolic function of black-sholes d1
def vol_parabolic(d1, vol_atm, skew, kurt):
    return  vol_atm + skew*(d1-1) + kurt*(d1-1)**2

# Black-Scholes delta
def delta_bs(d1):
    return stats.norm.cdf(d1)

# calculate BS d1
def d1_bs(S, K, T, sigma):
    return (np.log(S/K) + (0.5*sigma**2)*T) / (sigma*np.sqrt(T))

# calculate BS d2
def d2_bs(S, K, T, sigma):
    return d1_bs(S, K, T, sigma) - sigma*np.sqrt(T)

# Black-Scholes price
def bs_price(S, K, T, sigma, r=0, call=True):
    d1 = d1_bs(S, K, T, sigma)
    d2 = d2_bs(S, K, T, sigma)
    if call:
        return S*stats.norm.cdf(d1) - K*np.exp(-r*T)*stats.norm.cdf(d2)
    else:
        return K*np.exp(-r*T)*stats.norm.cdf(-d2) - S*stats.norm.cdf(-d1)

In [48]:
# plot vols as a scatterplot: x = N(d1), y = vol
d1 = np.linspace(-3, 3, 600)    # d1 range: -inf, +inf
bs_delta = delta_bs(d1)    # Black-Scholes delta range: (0, 1)
vol = vol_parabolic(d1, 0.25, -0.05, 0.02)
fig = px.scatter(x=bs_delta, y=vol)
fig.update_layout(xaxis_title='N(d1)', yaxis_title='Volatility')
fig.show()

In [69]:
strike_pct = 1.005
vol_atm = 0.25
skew = -0.05
kurt = 0.02


In [70]:
# add option price to df_trade, for selected strike, defined as percentage of spot
call = True if strike_pct > 1 else False
df_trade['Strike'] = df_trade['Close']*strike_pct

# calculate option price from BS formula
df_trade['Option_Price'] = df_trade.apply(lambda row: bs_price(row['Close'], row['Strike'], row['time_to_expiry'], vol_atm, call=call), axis=1)


In [71]:
if call:
    df_trade['Option_Payoff'] = df_trade.apply(lambda row: max(0, row['Close_At_Expiry'] - row['Strike']), axis=1)
else:
    df_trade['Option_Payoff'] = df_trade.apply(lambda row: max(0, row['Strike'] - row['Close_At_Expiry']), axis=1)

In [72]:
df_trade

,Close,Expiry,Close_At_Expiry,time_to_expiry,hours_to_exp,Strike,Option_Price,Option_Payoff
Date,,,,,,,,
2024-03-18 10:00:00-05:00,504.06,2024-03-18 14:55:00-05:00,502.05,0.001951,4.916667,506.58030,1.189251,0.00000
2024-03-19 10:00:00-05:00,501.65,2024-03-19 14:55:00-05:00,504.88,0.001951,4.916667,504.15825,1.183565,0.72175
2024-03-20 10:00:00-05:00,505.20,2024-03-20 14:55:00-05:00,509.44,0.001951,4.916667,507.72600,1.191940,1.71400
2024-03-21 10:00:00-05:00,512.33,2024-03-21 14:55:00-05:00,511.10,0.001951,4.916667,514.89165,1.208762,0.00000
2024-03-22 10:00:00-05:00,510.45,2024-03-22 14:55:00-05:00,510.28,0.001951,4.916667,513.00225,1.204327,0.00000
...,...,...,...,...,...,...,...,...
2026-03-11 10:00:00-05:00,677.85,2026-03-11 14:55:00-05:00,676.35,0.001951,4.916667,681.23925,1.599281,0.00000
2026-03-12 10:00:00-05:00,668.15,2026-03-12 14:55:00-05:00,666.05,0.001951,4.916667,671.49075,1.576395,0.00000
2026-03-13 10:00:00-05:00,665.10,2026-03-13 14:55:00-05:00,662.31,0.001951,4.916667,668.42550,1.569199,0.00000


In [73]:
df_trade['PNL'] = df_trade['Option_Payoff'] - df_trade['Option_Price']

In [74]:
# calculate cumulative PNL, SHarpe, drawdown, etc.
df_trade['Cumulative_PNL'] = df_trade['PNL'].cumsum()
df_trade['Cumulative_Return'] = df_trade['Cumulative_PNL'] / df_trade['Close']
df_trade['Sharpe'] = df_trade['Cumulative_Return'].mean() / df_trade['Cumulative_Return'].std() * np.sqrt(252)
df_trade['Drawdown'] = df_trade['Cumulative_Return'].cummax() - df_trade['Cumulative_Return']
df_trade['Max_Drawdown'] = df_trade['Drawdown'].min()
df_trade

,Close,Expiry,Close_At_Expiry,time_to_expiry,hours_to_exp,Strike,Option_Price,Option_Payoff,PNL,Cumulative_PNL,Cumulative_Return,Sharpe,Drawdown,Max_Drawdown
Date,,,,,,,,,,,,,,
2024-03-18 10:00:00-05:00,504.06,2024-03-18 14:55:00-05:00,502.05,0.001951,4.916667,506.58030,1.189251,0.00000,-1.189251,-1.189251,-0.002359,-31.017065,0.000000,0.0
2024-03-19 10:00:00-05:00,501.65,2024-03-19 14:55:00-05:00,504.88,0.001951,4.916667,504.15825,1.183565,0.72175,-0.461815,-1.651065,-0.003291,-31.017065,0.000932,0.0
2024-03-20 10:00:00-05:00,505.20,2024-03-20 14:55:00-05:00,509.44,0.001951,4.916667,507.72600,1.191940,1.71400,0.522060,-1.129006,-0.002235,-31.017065,0.000000,0.0
2024-03-21 10:00:00-05:00,512.33,2024-03-21 14:55:00-05:00,511.10,0.001951,4.916667,514.89165,1.208762,0.00000,-1.208762,-2.337768,-0.004563,-31.017065,0.002328,0.0
2024-03-22 10:00:00-05:00,510.45,2024-03-22 14:55:00-05:00,510.28,0.001951,4.916667,513.00225,1.204327,0.00000,-1.204327,-3.542095,-0.006939,-31.017065,0.004704,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-11 10:00:00-05:00,677.85,2026-03-11 14:55:00-05:00,676.35,0.001951,4.916667,681.23925,1.599281,0.00000,-1.599281,-578.052853,-0.852774,-31.017065,0.850539,0.0
2026-03-12 10:00:00-05:00,668.15,2026-03-12 14:55:00-05:00,666.05,0.001951,4.916667,671.49075,1.576395,0.00000,-1.576395,-579.629248,-0.867514,-31.017065,0.865279,0.0
2026-03-13 10:00:00-05:00,665.10,2026-03-13 14:55:00-05:00,662.31,0.001951,4.916667,668.42550,1.569199,0.00000,-1.569199,-581.198448,-0.873851,-31.017065,0.871616,0.0
